# 10 scenarios transition limits — 遷移・限界・resilient

**対象:** お客様（MPC 設計経験者）との **理論・数式・パラメータ** ディスカッション  
**Part 4/4** — Scenario 16–20

各シナリオは **路面 · 速度 · 勾配 · 実装** を結びつけています。  
理論の前提: [00_theory_grf_mpc_wbc.ipynb](./00_theory_grf_mpc_wbc.ipynb)  
QA 索引: [11_qa_discussion_master.ipynb](./11_qa_discussion_master.ipynb)

```bash
python scripts/scenario_labs.py --list
python scripts/scenario_labs.py --scenario sc16_uphill_to_downhill_switch
```


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

from tuning_labs import (
    TUNING_LABS,
    list_labs,
    run_lab,
    run_lab_pair,
    plot_speed_trial_journey,
    plot_param_study_mu,
    load_cached_lab_results,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")

from scenario_labs import (
    SCENARIO_LABS,
    compare_preset_table,
    run_scenario,
    run_scenario_pair,
    scenario_table,
)


## Scenario 16 — 上り→下り — preset 切替マトリクス

| 項目 | 内容 |
|------|------|
| **ID** | `sc16_uphill_to_downhill_switch` |
| **分類** | transition / expert |
| **路面** | bumpy_uphill → bumpy_downhill |
| **速度** | 5.0 kph |
| **勾配** | mixed |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

同一 5 kph でも上り→下りで μ, duty, ramp, ref_z がすべて変わる。

### 理論（Layer 2–3）

3 層パイプラインは同じ。変わるのは $\theta$ と必要 GRF 配分。

$$\Delta F_{ix}^{req} \approx mg(\sin\theta_{up} - \sin\theta_{down})$$

### パラメータ焦点

| パラメータ | uphill | downhill |
| mu | 0.38 | 0.35 |
| duty | 0.78 | 0.82 |
| ramp | 20s | 22s |

**実装:** YAML diff + speed_terrain_results.json

### ノウハウ

configs/pympc_presets/session04_bumpy_*.yaml を並べて diff。

### 議論用 Q&A

Q: 1 つの adaptive MPC で全部やれない？
A: 可能だが gain scheduling / 地形推定が必要。まず固定 preset で理解。


In [ ]:
from scenario_labs import compare_preset_table
import pandas as pd

df = pd.DataFrame(compare_preset_table(['session04_bumpy_uphill', 'session04_bumpy_flat', 'session04_bumpy_downhill']))
display(df)


## Scenario 17 — 7 kph 挑戦 — 速度限界の探索

| 項目 | 内容 |
|------|------|
| **ID** | `sc17_bumpy_7kph` |
| **分類** | speed / expert |
| **路面** | bumpy_flat |
| **速度** | 7.0 kph |
| **勾配** | flat + 凸凹 |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

5 kph 勝者 preset を 7 kph に上げると no-fall はほぼ不可能。

### 理論（Layer 2–3）

$v^{ref}$↑ → 同 $\mu$ で $|F_{ix}|$ 不足または円錐違反。

$$a_x^{des} = \dot{v}^{ref} \text{ or steady } v^{ref}/T, \quad \sum F_{ix} \approx m a_x$$

### パラメータ焦点

| `target_speed_kph` | 7.0 | 限界探索 |

**実装:** 7 kph fail vs 5 kph baseline

### ノウハウ

7 kph は本 workshop 範囲外。5 kph 安定後に μ↑ ramp↓ で段階探索。

### 議論用 Q&A

Q: mean_kph 4.0 で success 判定の意味は？
A: 指令 5 kph に対し 4 kph 以上で『実用追従』と定義。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc17_bumpy_7kph")
fig = compare_runs(pair)
plt.suptitle("Scenario 17: 7 kph 挑戦 — 速度限界の探索", y=1.02)
plt.show()


## Scenario 18 — 上り ref_z 不足 — 勾配+低 CoM

| 項目 | 内容 |
|------|------|
| **ID** | `sc18_uphill_ref_z_low` |
| **分類** | slope / advanced |
| **路面** | bumpy_uphill |
| **速度** | 5.0 kph |
| **勾配** | uphill |
| **preset** | `session04_bumpy_uphill` |

### シナリオ

上りで ref_z_scale=1.02 では足が地面に引っかかる。1.08–1.10 が必要。

### 理論（Layer 2–3）

$z^{ref}$↓ → WBC が脚縮小 → 勾配変化で toe collision。

$$h_{CoM}^{ref} = ref_z, \quad h_{CoM}^{ref} \uparrow \Rightarrow \text{ground clearance} \uparrow$$

### パラメータ焦点

| `ref_z_scale` | 1.02 → 1.08 | 上り向け |

**実装:** ref_z_scale 単独 A/B on uphill

### ノウハウ

上り・下りとも ref_z やや↑。下りは 1.10 まで。

### 議論用 Q&A

Q: ref_z と duty の関係？
A: 独立だが両方とも支持安定性に効く。上りは両方やや↑。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc18_uphill_ref_z_low")
fig = compare_runs(pair)
plt.suptitle("Scenario 18: 上り ref_z 不足 — 勾配+低 CoM", y=1.02)
plt.show()


## Scenario 19 — resilient 20 m — no-fall 不可でも学習

| 項目 | 内容 |
|------|------|
| **ID** | `sc19_resilient_flat_win` |
| **分類** | speed / advanced |
| **路面** | bumpy_flat |
| **速度** | 5.0 kph |
| **勾配** | flat + 凸凹 |
| **preset** | `session04_bumpy_flat` |

### シナリオ

転倒 reset しながら累積 20 m。17 falls で成功（Session 4 勝者）。

### 理論（Layer 2–3）

評価関数を distance 累積に。制御則は同一 SRB-MPC。

$$\max \sum distance \quad \text{s.t. falls} \le N_{max}$$

### パラメータ焦点

| `max_falls` | 22 | resilient 許容 |

**実装:** load speed_terrain_results.json でキャッシュ表示可

### ノウハウ

デモ GIF は resilient 走行。議論は falls 数も必ず出す。

### 議論用 Q&A

Q: resilient はカンニング？
A: 指令追従・gait 設計の評価には有効。本番は falls=0 が目標。


In [ ]:
from scenario_labs import run_scenario

r = run_scenario("sc19_resilient_flat_win")
res = r.get("result", {})
for k in ("distance_m", "mean_kph", "success", "terminated", "falls", "mean_vx", "max_roll_deg"):
    if k in res:
        print(f"{k}: {res[k]}")


## Scenario 20 — 設計空間 — μ × duty × ramp × ref_z

| 項目 | 内容 |
|------|------|
| **ID** | `sc20_tradeoff_matrix` |
| **分類** | transition / expert |
| **路面** | 全地形（表形式） |
| **速度** | 5.0 kph |
| **勾配** | all |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

20 シナリオを貫くトレードオフ表。お客様 QA の索引。

### 理論（Layer 2–3）

Convex 近似: $s_i(k)$ 固定下で GRF $u$ について QP。

$$\min \|x-x^{ref}\|_Q + \|u\|_R \;\; \text{s.t.}\;\; |F_t|\le\mu F_z,\; F_z^{min}\le F_z\le F_z^{max}$$

### パラメータ焦点

| 症状 | 第一 | 第二 | 第三 |
| 即転倒 | ref_z↑ | freq↓ | μ↓ |
| 加速不足 | μ↑ | ramp↓ | grf_max↑ |
| 下り失敗 | duty↑ | μ↓ | ramp↑ |

**実装:** Notebook 11 QA マスターと連携

### ノウハウ

1 パラメータずつ A/B。YAML で勝ちパターンを資産化。

### 議論用 Q&A

Q: ADAS 操舵 MPC との対応は？
A: SRB≈車両、GRF≈タイヤ力、μ≈路面摩擦、WBC≈下位アクチュエータ。


In [ ]:
from scenario_labs import scenario_table
import pandas as pd

display(pd.DataFrame(scenario_table()))


---

## Part 4 チェックリスト

- [ ] Scenario 16–20 それぞれ **数式 → パラメータ → 結果** を説明できる  
- [ ] fail / OK の差が **摩擦円錐 · gait · 指令 ramp** のどれか特定できる  
- [ ] `configs/pympc_presets/` の YAML と対応づけられる  

**次:** [11_qa_discussion_master.ipynb](./11_qa_discussion_master.ipynb)
